# Model Evaluation and Selection

Experiments with train / cross-validation / test splits, polynomial regression, neural-network model comparison, and classification error.

**Note:** This notebook depends on additional data and helper files.


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler , PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import utils



In [ ]:
##Regression ##
data = np.loadtxt("data_w3_ex1.csv",delimiter = ',')
x = data[:,0]
y = data[:,1]

x = x.reshape(-1,1)
y = y.reshape(-1,1)

print(x.shape)
print(y.shape)

In [ ]:
utils.plot_dataset(x=x, y=y, title="input vs. target")

In [ ]:
x_train ,x_temp ,y_train , y_temp = train_test_split(x,y,test_size = 0.4 ,random_state = 1)

x_cv , x_test ,y_cv,y_test = train_test_split(x_temp,y_temp,test_size = 0.5 ,random_state =1  )

del x_temp , y_temp

print(x_train.shape)
print(y_train.shape)

print(x_cv.shape)
print(y_cv.shape)

print(x_test.shape)
print(y_test.shape)

In [ ]:
utils.plot_train_cv_test(x_train, y_train, x_cv, y_cv, x_test, y_test, title="input vs. target")

In [ ]:
##Feature Scaling ##
scaler_linear = StandardScaler()
x_train_scaled = scaler_linear.fit_transform(x_train)

print(f"Computed mean of the training set: {scaler_linear.mean_.squeeze():.2f}")
print(f"Computed standard deviation of the training set: {scaler_linear.scale_.squeeze():.2f}")

# Plot the results
utils.plot_dataset(x=x_train_scaled, y=y_train, title="scaled input vs. target")

In [ ]:
linear_model  = LinearRegression()
linear_model.fit(x_train_scaled,y_train)

In [ ]:
##MSE using sklearn ##
y_hat = linear_model.predict(x_train_scaled)
print(f"training MSE (using sklearn function) : {mean_squared_error( y_train,y_hat)/2}")

In [ ]:
## MSE loop implementation ##
total_squared_error = 0
for i in range(len(y_hat)):
  total_squared_error_i = (y_hat[i] - y_train[i])**2
  total_squared_error += total_squared_error_i
mse = total_squared_error /(2*len(y_hat))

print(f"training mse (for loop implemetation) : {mse.squeeze()}")



In [ ]:
x_cv_scaled = scaler_linear.transform(x_cv)
print(f"mean used to scale the CV set : {scaler_linear.mean_.squeeze():.2f}")
print(f"std used to scale the CV set : {scaler_linear.scale_.squeeze():.2f}")

y_hat = linear_model.predict(x_cv_scaled)

print(f"cross validation MSE : {mean_squared_error(y_cv , y_hat) / 2}")



In [ ]:
### Adding Polynomial Features ###

In [ ]:
poly = PolynomialFeatures(degree=2 , include_bias=False)
## Compute the number of features and transform the training set
x_train_mapped = poly.fit_transform(x_train)

print(x_train_mapped[:5])


In [ ]:
scaler_poly = StandardScaler()

x_train_mapped_scaled = scaler_poly.fit_transform(x_train_mapped)

print(x_train_mapped_scaled[:5])


In [ ]:
model = LinearRegression()
model.fit(x_train_mapped_scaled , y_train)

y_hat = model.predict(x_train_mapped_scaled)
print(f"Training MSE : {mean_squared_error(y_train , y_hat)/2}")


In [ ]:
x_cv_mapped = poly.transform(x_cv)
x_cv_mapped_scaled = scaler_poly.transform(x_cv_mapped)

y_hat = model.predict(x_cv_mapped_scaled)
print(f"cv MSE : {mean_squared_error(y_cv , y_hat)/2}")



In [ ]:
train_mses = []
cv_mses = []
models = []
polys = []
scalers = []
for degree in range(1,11):
  print("degree:",degree)
  poly = PolynomialFeatures(degree = degree , include_bias = False )
  x_train_mapped = poly.fit_transform(x_train)
  polys.append(poly)

  ## Scale the training set
  scaler_poly = StandardScaler()
  x_train_mapped_scaled = scaler_poly.fit_transform(x_train_mapped)
  scalers.append(scaler_poly)

  ## Create and train the model
  model = LinearRegression()
  model.fit(x_train_mapped_scaled,y_train)
  models.append(model)
  ## Computing the training MSE
  y_hat = model.predict(x_train_mapped_scaled)
  train_mse = mean_squared_error(y_train,y_hat)/2
  train_mses.append(train_mse)
  ## Add polynomial features and scale across validation set
  x_cv_mapped = poly.transform(x_cv)
  x_cv_mapped_scaled = scaler_poly.transform(x_cv_mapped)

  ## Computing cv errror MSE
  y_hat = model.predict(x_cv_mapped_scaled)
  cv_mse = mean_squared_error(y_cv,y_hat)/2
  cv_mses.append(cv_mse)

## plotting the results
degrees=range(1,11)
utils.plot_train_cv_mses(degrees, train_mses, cv_mses, title="degree of polynomial vs. train and CV MSEs")




In [ ]:
##Chosing the best model ##
degree = np.argmin(cv_mses)+1
print(f"Lowest CV MSE is found in the model with degree = {degree}")

In [ ]:
x_test_mapped = polys[degree-1].transform(x_test)
x_test_mapped_scaled = scalers[degree-1].transform(x_test_mapped)
y_hat = models[degree-1].predict(x_test_mapped_scaled)
mse_test =  mean_squared_error(y_test,y_hat) / 2
print(f'Training MSE : {train_mses[degree-1]:.2f}')
print(f'cross validation MSE : {cv_mses[degree-1]:.2f}')
print(f'test MSE : {mse_test:.2f}')

 **NEURAL NETWORKS**

In [ ]:
degree = 1
poly = PolynomialFeatures(degree=1 , include_bias=False)
x_train_mapped = poly.fit_transform(x_train)
x_cv_mapped = poly.transform(x_cv)
x_test_mapped = poly.transform(x_test)

In [ ]:
scaler = StandardScaler()
x_train_mapped_scaled = scaler.fit_transform(x_train_mapped)
x_cv_mapped_scaled = scaler.transform(x_cv_mapped)
x_test_mapped_scaled = scaler.transform(x_test_mapped)


In [ ]:
nn_models = utils.build_models()

for model in nn_models:

    model.compile(
        loss="mse",
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.1)
    )

    history = model.fit(
        x_train_mapped_scaled,
        y_train,
        epochs=300,
        verbose=0
    )

    print(model.name)
    print("first loss:", history.history["loss"][0])
    print("final loss:", history.history["loss"][-1])

In [ ]:
for model in nn_models:
    pred = model.predict(x_train_mapped_scaled, verbose=0)

    print(model.name)
    print(pred[:5].flatten())

In [ ]:
nn_train_mses = []
nn_cv_mses = []

nn_models = utils.build_models()

for model in nn_models:

    model.compile(
        loss="mse",
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.1)
    )

    model.fit(
        x_train_mapped_scaled,
        y_train,
        epochs=300,
        verbose=0
    )

    # training
    y_hat = model.predict(x_train_mapped_scaled, verbose=0)
    train_mse = mean_squared_error(y_train, y_hat) / 2
    nn_train_mses.append(train_mse)

    # CV
    y_hat = model.predict(x_cv_mapped_scaled, verbose=0)
    cv_mse = mean_squared_error(y_cv, y_hat) / 2
    nn_cv_mses.append(cv_mse)

for i in range(len(nn_models)):
    print(
        f"Model {i+1}: "
        f"train={nn_train_mses[i]:.2f}, "
        f"cv={nn_cv_mses[i]:.2f}"
    )

In [ ]:
model_num = 3
y_hat = nn_models[model_num - 1].predict(x_test_mapped_scaled)
test_mse = mean_squared_error(y_test,y_hat)/2

print(f"Selected Model: {model_num}")
print(f"Training MSE: {nn_train_mses[model_num-1]:.2f}")
print(f"Cross Validation MSE: {nn_cv_mses[model_num-1]:.2f}")
print(f"Test MSE: {test_mse:.2f}")

**CLASSIFICATION**

In [ ]:
data = np.loadtxt('data_w3_ex2.csv',delimiter = ',')

x_bc = data[:,:-1]
y_bc = data[:,-1]

y_bc = y_bc.reshape(-1,1)

print(x_bc.shape)
print(y_bc.shape)

In [ ]:
utils.plot_bc_dataset(x=x_bc, y=y_bc, title="x1 vs. x2")

In [ ]:
from sklearn.model_selection import train_test_split

x_bc_train , x_temp , y_bc_train , y_temp = train_test_split(x_bc,y_bc,test_size = 0.4,random_state = 1)

x_bc_cv , x_bc_test , y_bc_cv, y_bc_test = train_test_split(x_temp , y_temp ,test_size = 0.5,random_state = 1)

del x_temp , y_temp

print(x_bc_train.shape)
print(y_bc_train.shape)
print(x_bc_cv.shape)
print(y_bc_cv.shape)
print(x_bc_test.shape)
print(y_bc_test.shape )


In [ ]:
scaler_linear = StandardScaler()
x_bc_train_scaled = scaler_linear.fit_transform(x_bc_train)
x_bc_cv_scaled = scaler_linear.transform(x_bc_cv)
x_bc_test_scaled = scaler_linear.transform(x_bc_test)


In [ ]:
## Toy example ##
probabilities= np.array([0.2 , 0.6 , 0.7 , 0.3 , 0.8 ])
predictions = np.where(probabilities > 0.5 , 1, 0)

ground_truth = np.array([1,1,1,1,1])

misclassified  =  0
num_predictions = len(predictions)

for i in range(len(probabilities)):
  if predictions[i] != ground_truth[i] :
    misclassified += 1

fraction_error = misclassified / num_predictions

print(f"probabilities: {probabilities}")
print(f"predictions with threshold=0.5: {predictions}")
print(f"targets: {ground_truth}")
print(f"fraction of misclassified data (for-loop): {fraction_error}")

print(f"fraction of misclassified data (with np.mean()): {np.mean(predictions != ground_truth)}")




In [ ]:
nn_train_error = []
nn_cv_error = []

models_bc = utils.build_models()

for model in models_bc :
  model.compile(
      loss = tf.keras.losses.BinaryCrossentropy(from_logits=True),
      optimizer = tf.keras.optimizers.Adam(learning_rate = 0.01),
  )
  model.fit(x_bc_train_scaled ,y_bc_train,epochs =200 ,verbose =0)

  print("\n Done")

  ## Set the threshhold for classification
  threshold = 0.5
  ## Record the fraction of misclassified examples for the training set
  y_hat = model.predict(x_bc_train_scaled)
  y_hat = tf.nn.sigmoid(y_hat)
  y_hat = np.where(y_hat >= threshold ,1,0)
  train_error = np.mean(y_hat != y_bc_train)
  nn_train_error.append(train_error)

  ## Record The fraction of misclassified examples for cross validation set
  y_hat = model.predict(x_bc_cv_scaled)
  y_hat = tf.nn.sigmoid(y_hat)
  y_hat = np.where(y_hat >= threshold ,1,0)
  cv_error = np.mean(y_hat != y_bc_cv)
  nn_cv_error.append(cv_error)

  for model_num in range(len(nn_train_error)):
    print(f"Model {model_num+1} : Training Set classification Error :{nn_train_error[model_num]:.2f}," +
          f"CV set classification error : {nn_cv_error[model_num] : .2f}")






In [ ]:
model_num = 1
y_hat = models_bc[model_num - 1].predict(x_bc_test_scaled)
y_hat = tf.nn.sigmoid(y_hat)
y_hat = np.where(y_hat >= threshold , 1,0)
test_error = np.mean(y_hat != y_bc_test)
print(f'Selected Model : {model_num}')
print(f"Training Set Classification Error: {nn_train_error[model_num-1]:.4f}")
print(f"CV Set Classification Error: {nn_cv_error[model_num-1]:.4f}")
print(f"Test Set Classification Error: {test_error:.4f}")